In [20]:
# Carga del conjunto de datos

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from pickle import dump

url = "https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
df.head()

Shape: (891, 3)


,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0


In [21]:
# Distribución de la variable objetivo
print("Distribución de clases:")
print(df["polarity"].value_counts())
print(f"\n→ {df['polarity'].value_counts()[0]} reseñas negativas (0)")
print(f"→ {df['polarity'].value_counts()[1]} reseñas positivas (1)")

Distribución de clases:
polarity
0    584
1    307
Name: count, dtype: int64

→ 584 reseñas negativas (0)
→ 307 reseñas positivas (1)


In [22]:
# Eliminar package_name: no aporta información sobre el sentimiento
df = df.drop(columns=["package_name"])

# Eliminar filas con texto nulo
df = df.dropna(subset=["review"])

print("Shape tras limpieza:", df.shape)

Shape tras limpieza: (891, 2)


In [23]:
# Limpiar texto — minúsculas y sin espacios extra
df["review"] = df["review"].str.strip().str.lower()

print("Ejemplo de reseña procesada:")
print(df["review"].iloc[0])

Ejemplo de reseña procesada:
privacy at least put some option appear offline. i mean for some people like me it's a big pressure to be seen online like you need to response on every message or else you be called seenzone only. if only i wanna do on facebook is to read on my newsfeed and just wanna response on message i want to. pls reconsidered my review. i tried to turn off chat but still can see me as online.


In [24]:
# Split train/test
X = df["review"]
y = df["polarity"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Train: {X_train.shape[0]} muestras")
print(f"Test:  {X_test.shape[0]} muestras")

Train: 712 muestras
Test:  179 muestras


In [25]:
# Transformar texto a matriz de conteos con CountVectorizer
# stop_words="english" elimina palabras sin significado (the, is, a...)
vec_model = CountVectorizer(stop_words="english")

# fit_transform en TRAIN: aprende el vocabulario Y transforma
X_train_vec = vec_model.fit_transform(X_train).toarray()

# transform en TEST: solo transforma (con el vocabulario aprendido del train)
X_test_vec = vec_model.transform(X_test).toarray()

print(f"Vocabulario aprendido: {len(vec_model.vocabulary_)} palabras únicas")
print(f"Shape de X_train_vec: {X_train_vec.shape}")
print(f"Shape de X_test_vec:  {X_test_vec.shape}")

Vocabulario aprendido: 3310 palabras únicas
Shape de X_train_vec: (712, 3310)
Shape de X_test_vec:  (179, 3310)


In [26]:
# Comparación de las tres implementaciones
# GaussianNB — diseñado para datos continuos (distribución normal)
model_gnb = GaussianNB()
model_gnb.fit(X_train_vec, y_train)
y_pred_gnb = model_gnb.predict(X_test_vec)
acc_gnb = accuracy_score(y_test, y_pred_gnb)
print(f"GaussianNB  accuracy: {acc_gnb:.4f}")

# MultinomialNB — diseñado para conteos/frecuencias discretas ✅
model_mnb = MultinomialNB()
model_mnb.fit(X_train_vec, y_train)
y_pred_mnb = model_mnb.predict(X_test_vec)
acc_mnb = accuracy_score(y_test, y_pred_mnb)
print(f"MultinomialNB accuracy: {acc_mnb:.4f}")

# BernoulliNB — diseñado para datos binarios (presencia/ausencia)
model_bnb = BernoulliNB()
model_bnb.fit(X_train_vec, y_train)
y_pred_bnb = model_bnb.predict(X_test_vec)
acc_bnb = accuracy_score(y_test, y_pred_bnb)
print(f"BernoulliNB  accuracy: {acc_bnb:.4f}")

print("\n→ Mejor modelo:", max([
    ("GaussianNB", acc_gnb),
    ("MultinomialNB", acc_mnb),
    ("BernoulliNB", acc_bnb)
], key=lambda x: x[1]))

GaussianNB  accuracy: 0.8045
MultinomialNB accuracy: 0.8156
BernoulliNB  accuracy: 0.7709

→ Mejor modelo: ('MultinomialNB', 0.8156424581005587)


In [27]:
# Random Forest 
model_rf = RandomForestClassifier(n_estimators=100, random_state=42)
model_rf.fit(X_train_vec, y_train)
y_pred_rf = model_rf.predict(X_test_vec)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"Random Forest accuracy: {acc_rf:.4f}")
print(f"MultinomialNB accuracy: {acc_mnb:.4f}")


Random Forest accuracy: 0.7989
MultinomialNB accuracy: 0.8156


In [28]:
# Guardar mejor modelo Naive Bayes (MultinomialNB, configuración default)
dump(model_mnb, open("../models/naive_bayes_multinomial_default.sav", "wb"))
print("✅ Modelo guardado como: naive_bayes_multinomial_default.sav")

# Guardar CountVectorizer 
dump(vec_model, open("../models/count_vectorizer.sav", "wb"))
print("✅ Vectorizador guardado como: count_vectorizer.sav")

✅ Modelo guardado como: naive_bayes_multinomial_default.sav
✅ Vectorizador guardado como: count_vectorizer.sav


In [29]:
# Otras alternativas: Regresión Logística

from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000, random_state=42)
model_lr.fit(X_train_vec, y_train)
y_pred_lr = model_lr.predict(X_test_vec)
acc_lr = accuracy_score(y_test, y_pred_lr)

print(f"Logistic Regression accuracy: {acc_lr:.4f}")

print("\n--- Resumen final ---")
resultados = [
    ("GaussianNB",           acc_gnb),
    ("MultinomialNB",        acc_mnb),
    ("BernoulliNB",          acc_bnb),
    ("Random Forest",        acc_rf),
    ("Logistic Regression",  acc_lr),
]
for nombre, acc in sorted(resultados, key=lambda x: x[1], reverse=True):
    print(f"  {nombre:<25} {acc:.4f}")

Logistic Regression accuracy: 0.8324

--- Resumen final ---
  Logistic Regression       0.8324
  MultinomialNB             0.8156
  GaussianNB                0.8045
  Random Forest             0.7989
  BernoulliNB               0.7709
